### Robot Tracking

This notebook contains the work I used to finetune the chosen tracking algorithm (Botsort). The goal of fine tuning is to only identify 6 robots per video and assign the correct IDs to the robots. 

In [15]:
!pip install scipy

In [29]:
import os
from pathlib import Path
from scipy.optimize import linear_sum_assignment

REPO_ROOT = Path(os.getcwd()).parent  

In [30]:
# Finding path to best model
for p in REPO_ROOT.rglob("best_tuned_yolov8.pt"):
    print(p)

/work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/yolov8_model/best_tuned_yolov8.pt


In [ ]:
VIDEO_NAME = 'cropped_Qualification 45 - 2025 Central Missouri Regional.mp4'
VIDEO_PATH = "/work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4"

# Change Model Path to be best model from above in line 2! 
MODEL_PATH = REPO_ROOT / "yolov8_model" / "best_tuned_yolov8.pt"

# Custom tracker 
CUSTOM_TRACKER_PATH = REPO_ROOT / "trackers" / "botsort_custom.yaml"

# Checking to see if file exist! 
print("Model:", MODEL_PATH)
print("Video:", VIDEO_PATH)
print("Tracker:", CUSTOM_TRACKER_PATH)

print("Model exists:", MODEL_PATH.exists())

Model: /work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/yolov8_model/best_tuned_yolov8.pt
Video: /work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4
Tracker: /work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/trackers/botsort_custom.yaml
Model exists: True


In [ ]:
# Not currently used but manuallt defining robot IDs to track robots 
TEAM_NUMBERS = [8825, 1736, 2357, 3928, 5809, 9570]  # Qualification 45 - Central Missouri Regional 2025 

In [ ]:
import numpy as np 

MAX_ROBOTS = 6
ROBOT_CLASS_ID = 1         # ID assigned by Yolo for Robots 
MAX_MISSING_FRAMES = 300
MAX_ASSIGNMENT_DIST = 200

robot_slots = {
    rid: {
        "position": None,
        "last_seen": -1,
        "active": False
    }
    for rid in ROBOT_IDS
}

def get_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1+x2)/2, (y1+y2)/2])

def distance(a, b):
    return np.linalg.norm(a-b)

def assign_robot_slots(boxes, frame_id):

    assignments = {}
    centers = [get_center(b) for b in boxes]

    for i, center in enumerate(centers):

        best_robot = None
        best_dist = float("inf")

        for rid, slot in robot_slots.items():

            if slot["position"] is None:
                best_robot = rid
                break

            d = distance(center, slot["position"])

            if d < best_dist:
                best_dist = d
                best_robot = rid

        # Only allow reassignment if detection is close enough
        if best_dist < MAX_ASSIGNMENT_DIST or not robot_slots[best_robot]["active"]:

            assignments[i] = best_robot

            robot_slots[best_robot]["position"] = center
            robot_slots[best_robot]["last_seen"] = frame_id
            robot_slots[best_robot]["active"] = True

    return assignments

In [34]:
from ultralytics import YOLO
import numpy as np
from collections import defaultdict
import cv2

print("Loading model...")
model = YOLO(MODEL_PATH)

def run_botsort(model_path, video_path, tracker_path, save_video=False, output_dir=None):

    print("Running BotSort tracking...")

    results = model.track(
        source=video_path,
        tracker=tracker_path,
        stream=True,
        persist=False,
        conf=0.35,
        device=0,
        verbose=False
    )

    tracking_results = []

    video_writer = None

    for frame_i, result in enumerate(results):

        if result.boxes.xyxy is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)

        # Keep only robots
        robot_indices = [i for i, c in enumerate(classes) if c == ROBOT_CLASS_ID]

        boxes = boxes[robot_indices]

        assignments = assign_robot_slots(boxes, frame_i)
        frame = result.plot()

        for i, robot_id in assignments.items():

            x1,y1,x2,y2 = boxes[i]

            cv2.putText(
                frame,
                f"Robot {robot_id}",
                (int(x1), int(y1)-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0,255,0),
                2
            )

        if save_video:
            if video_writer is None:

                h, w = frame.shape[:2]
                output_path = output_dir / "botsort_with_mapping.mp4"

                video_writer = cv2.VideoWriter(
                    str(output_path),
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    30,
                    (w, h)
                )

            video_writer.write(frame)

    if video_writer:
        video_writer.release()

    print("Tracking complete.")
    return tracking_results

    print("Tracking finished. Number of Unique IDs:", len(unique_ids))
    return tracking_results  

Loading model...


In [35]:
# Running BotSort
output_dir = REPO_ROOT / "tracking_output"
output_dir.mkdir(exist_ok=True)

botsort_results = run_botsort(
    MODEL_PATH, 
    VIDEO_PATH, 
    CUSTOM_TRACKER_PATH,
    save_video=True,
    output_dir=output_dir
)

Running BotSort tracking...
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
Tracking complete.


In [36]:
import pandas as pd

track_lifetimes = defaultdict(int)
track_first_frame = {}
track_last_frame = {}

detections_per_frame = []
all_ids = set()

for frame_i, r in enumerate(botsort_results):

    if r.boxes.id is None:
        detections_per_frame.append(0)
        continue

    ids = r.boxes.id.cpu().numpy().astype(int)
    detections_per_frame.append(len(ids))

    for tid in ids:

        all_ids.add(tid)

        if tid not in track_first_frame:
            track_first_frame[tid] = frame_i

        track_last_frame[tid] = frame_i
        track_lifetimes[tid] += 1


lifetimes = list(track_lifetimes.values())

fragmentations = 0
for tid in track_first_frame:

    expected = track_last_frame[tid] - track_first_frame[tid] + 1
    actual = track_lifetimes[tid]

    fragmentations += max(expected - actual, 0)


metrics = {
    "Total Frames": len(botsort_results),
    "Total Unique Track IDs": len(all_ids),
    "Max Robots In Frame": max(detections_per_frame) if detections_per_frame else 0,
    "Avg Robots Per Frame": round(np.mean(detections_per_frame), 2) if detections_per_frame else 0,
    "Frames With 6 Robots": sum(1 for x in detections_per_frame if x == 6),
    "Avg Track Lifetime (frames)": round(np.mean(lifetimes), 2) if lifetimes else 0,
    "Longest Track Lifetime": max(lifetimes) if lifetimes else 0,
    "Track Fragmentation Score": fragmentations
}

pd.DataFrame(metrics.items(), columns=["Metric", "Value"])

,Metric,Value
0,Total Frames,0
1,Total Unique Track IDs,0
2,Max Robots In Frame,0
3,Avg Robots Per Frame,0
4,Frames With 6 Robots,0
5,Avg Track Lifetime (frames),0
6,Longest Track Lifetime,0
7,Track Fragmentation Score,0
